# Merkle–Damgård Hash Function Implementation

## Overview
This notebook demonstrates a simple, conceptual implementation of a cryptographic hash function built using the **Merkle–Damgård construction**. 

The pipeline processes input messages through the following stages:
1. **Rotation Helper**: Handles circular bitwise shifting.
2. **Compression Function**: Combines the intermediate hash state with a message block using XOR and rotation.
3. **Padding (Merkle–Damgård Strengthening)**: Formats the message to a fixed block boundary and appends the bit-length.
4. **Hashing Core**: Processes the padded message iteratively block-by-block.
5. **Output Formatting**: Formats the final 64-bit integer into a fixed-length hexadecimal string.

## 1. Bitwise Rotation & Compression Function

* **`rotate_left(x, bits, width=64)`**: Implements circular left bit rotation on a 64-bit integer. Bits shifted past the 64th position on the left wrap around to the least significant positions on the right.
* **`compression(h, m)`**: Represents the core state update function $f(h, m)$. It combines the current 64-bit state $h$ with the message block $m$ via a bitwise XOR operation (`h ^ m`) and applies a 7-bit left rotation to introduce diffusion.

In [1]:
import struct

def rotate_left(x, bits, width=64):
    return ((x << bits) | (x >> (width - bits))) & ((1 << width) - 1)

# Toy compression function: XOR and rotate
def compression(h, m):
    return rotate_left(h ^ m, 7)

## 2. Message Padding (Merkle–Damgård Strengthening)

Standard Merkle–Damgård construction requires input messages to be padded so that their length is a multiple of the block size (8 bytes / 64 bits here).

* **`pad_message(msg_bytes, block_size=8)`**:
  1. **Bit Length Calculation**: Saves `msg_len_bits` to preserve the original message size before padding.
  2. **Delimiter Insertion**: Appends `b'\x80'` (binary `10000000`), marking the boundary between data and padding.
  3. **Zero-Padding**: Appends `b'\x00'` bytes until the remaining space in the final block is exactly 8 bytes (reserved for length).
  4. **Length Attachment**: Appends the 64-bit big-endian integer representation of `msg_len_bits` via `struct.pack('>Q', msg_len_bits)`.

In [2]:

# Pad message using Merkle–Damgård strengthening
def pad_message(msg_bytes, block_size=8):
    msg_len_bits = len(msg_bytes) * 8
    # Append '1' bit then zeroes
    padded = msg_bytes + b'\x80'
    while (len(padded) + 8) % block_size != 0:
        padded += b'\x00'
    # Append 64-bit length
    padded += struct.pack('>Q', msg_len_bits)
    return padded

## 3. Merkle–Damgård Hash Core & Hexadecimal Formatting

* **`merkle_damgard_hash(message, iv, block_size)`**:
  - Initializes state $h$ with the Initialization Vector `iv = 0x0123456789ABCDEF`.
  - Splits the padded message into 8-byte blocks.
  - Converts each block to a big-endian integer $m$ and updates the hash iteratively: $h_{i+1} = \text{compression}(h_i, m_i)$.
* **`hash_to_hex(h)`**: Converts the final 64-bit integer state into a 16-character hexadecimal string, zero-padded if necessary.

In [3]:
def merkle_damgard_hash(message: bytes, iv=0x0123456789ABCDEF, block_size=8):
    padded = pad_message(message, block_size)
    h = iv
    for i in range(0, len(padded), block_size):
        block = padded[i:i + block_size]
        m = int.from_bytes(block, 'big')
        h = compression(h, m)
    return h

def hash_to_hex(h: int) -> str:
    return hex(h)[2:].zfill(16)

## 4. Execution Example
Hashes a sample string and outputs the resulting 64-bit hex digest.

In [7]:
# Example usage
#msg = b"A un amigo perdido. Tu ya no debes recordarme en el pais extranjero"
msg = b"""The Tor Project was a creation of the state that ended up becoming 
one of the few effective shields against the state's surveillance. Tor is free 
and open-source software that, if used carefully, allows its users to browse 
online with the closest thing to perfect anonymity that can be practically achieved at scale.
(Snowden, Permanent Record)"""
digest = merkle_damgard_hash(msg)
print("Hash:", hash_to_hex(digest))

Hash: c41daecae86a978f
